<a href="https://colab.research.google.com/github/SamikshaSingh1904/Absorbance_Analyze/blob/main/thinkstruct_patent_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thinkstruct Patent Search Engine

**Problem Statement:** Build a semantic + hybrid search engine over USPTO vehicle patent applications (2024) that lets patent agents quickly find relevant prior art using natural-language queries, with optional filters on classification codes and keywords.

**Enhancements implemented:**
1. **Hybrid Searching** — classification-code prefix filter + keyword filter in title/abstract, with performance benchmarking
2. **Two-Phase Re-Ranking** — cross-encoder re-ranks the top-100 semantic candidates down to the final top-k, improving precision

**Data field handling decision:**  
Patents missing `claims` OR `doc_number` are excluded. Claims are the legally enforceable definition of an invention and are the most important field for similarity search; a patent without claims has no searchable substance. `doc_number` is the unique identifier required for deduplication and result display. All other fields (abstract, title, detailed_description, classification) default to empty strings or `'N/A'` if absent, rather than dropping those patents — their claims are still searchable.

---

## 0. Install dependencies

In [12]:
# Run once per Kaggle session
!pip install -q sentence-transformers faiss-cpu

In [13]:
from google.colab import files
uploaded = files.upload()

Saving patents_ipa240215.json to patents_ipa240215 (1).json
Saving patents_ipa240222.json to patents_ipa240222 (1).json
Saving patents_ipa240229.json to patents_ipa240229 (1).json
Saving patents_ipa240307.json to patents_ipa240307 (1).json
Saving patents_ipa240314.json to patents_ipa240314 (1).json
Saving patents_ipa240321.json to patents_ipa240321 (1).json
Saving patents_ipa240328.json to patents_ipa240328 (1).json
Saving patents_ipa240404.json to patents_ipa240404 (1).json
Saving patents_ipa240411.json to patents_ipa240411 (1).json
Saving patents_ipa240418.json to patents_ipa240418 (1).json
Saving patents_ipa240425.json to patents_ipa240425 (1).json
Saving patents_ipa240502.json to patents_ipa240502 (1).json
Saving patents_ipa240509.json to patents_ipa240509 (1).json
Saving patents_ipa240516.json to patents_ipa240516 (1).json
Saving patents_ipa240523.json to patents_ipa240523 (1).json
Saving patents_ipa240530.json to patents_ipa240530 (1).json
Saving patents_ipa240606.json to patents

## 1. Imports & configuration

In [14]:
import json
import os
import glob
import time
import re
import warnings
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR = '/content'

# ── Models ─────────────────────────────────────────────────────────────────
# Bi-encoder: fast embedding model used for the first-pass semantic search.
# 'all-MiniLM-L6-v2' is 80 MB and runs comfortably on CPU — good Kaggle default.
BI_ENCODER_MODEL  = 'all-MiniLM-L6-v2'

# Cross-encoder: slower but more accurate — used only to re-rank the top-100
# candidates from the bi-encoder. 'cross-encoder/ms-marco-MiniLM-L-6-v2'
# is a strong, compact model for relevance scoring.
CROSS_ENCODER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

# ── Search defaults ─────────────────────────────────────────────────────────
PHASE1_CANDIDATES = 100   # bi-encoder retrieves this many before re-ranking
DEFAULT_TOP_K     = 5     # final results returned to the user

print('Configuration loaded.')

Configuration loaded.


## 2. Data ingestion

In [15]:
def load_patents(directory: str) -> pd.DataFrame:
    """
    Load all patents_ipa*.json files from `directory` into a single DataFrame.

    Inclusion / exclusion decisions
    --------------------------------
    EXCLUDED  — patents missing 'claims' or 'doc_number':
        • claims     : the legally enforceable definition of an invention;
                       without them there is nothing meaningful to embed.
        • doc_number : the unique identifier; without it we cannot deduplicate
                       or reliably present results.

    INCLUDED with defaults — all other optional fields:
        • abstract, title, classification, detailed_description
          default to '' / 'N/A' so the patent's claims remain searchable.

    Embedding text
    --------------
    We concatenate abstract + all claims into a single `search_text` field.
    Rationale: the abstract gives the high-level invention summary while the
    claims carry the precise technical scope — both matter for similarity.
    """
    files = sorted(glob.glob(os.path.join(directory, 'patents_ipa*.json')))
    if not files:
        raise FileNotFoundError(
            f'No patents_ipa*.json files found in "{directory}".\n'
            'Please set DATA_DIR to the correct Kaggle Dataset path.'
        )

    records, skipped = [], 0
    for fp in files:
        with open(fp) as fh:
            batch = json.load(fh)
        for p in batch:
            # Hard requirement: claims and doc_number must be present
            if not p.get('claims') or not p.get('doc_number'):
                skipped += 1
                continue

            claims_raw = p['claims']
            claims_text = (
                ' '.join(claims_raw) if isinstance(claims_raw, list)
                else str(claims_raw)
            )

            abstract = p.get('abstract', '')

            # The text that gets embedded for semantic search
            search_text = f"{abstract} {claims_text}".strip()

            # Parse the CPC prefix (e.g. 'B60B104FI' -> 'B60B') for fast filtering
            raw_code = p.get('classification', '')
            # Classification codes are like 'B60B104FI' — extract the letter+digit prefix
            cpc_prefix = re.match(r'[A-Z]\d{2}[A-Z]', raw_code)
            cpc_prefix = cpc_prefix.group(0) if cpc_prefix else raw_code

            records.append({
                'doc_id'        : p['doc_number'],
                'title'         : p.get('title', 'N/A'),
                'abstract'      : abstract,
                'claims'        : claims_text,
                'classification': raw_code,
                'cpc_prefix'    : cpc_prefix,           # pre-parsed for fast filter
                'search_text'   : search_text,
                'source_file'   : os.path.basename(fp),
            })

    df = pd.DataFrame(records)
    print(f'Loaded {len(df):,} patents from {len(files)} files  |  Skipped {skipped} (missing claims or doc_number)')
    return df


df_patents = load_patents(DATA_DIR)
df_patents.head(2)

Loaded 1,280 patents from 128 files  |  Skipped 0 (missing claims or doc_number)


,doc_id,title,abstract,claims,classification,cpc_prefix,search_text,source_file
0,20240051333,SPOKE,A spoke includes an axle body and two connecti...,"an axle body, having a middle segment and two ...",B60B104FI,B60B,A spoke includes an axle body and two connecti...,patents_ipa240215 (1).json
1,20240051334,"INSERT, PROTECTION DEVICE, WHEEL AND VEHCILE","The present application relates to an insert, ...","a body, provided on the wheel rim and/or the w...",B60B706FI,B60B,"The present application relates to an insert, ...",patents_ipa240215 (1).json


## 3. Build the search engine

In [16]:
class ThinkstructSearch:
    """
    Two-phase hybrid patent search engine.

    Architecture
    ------------
    Phase 1 — Bi-encoder + FAISS (fast, approximate)
        • Encodes every patent's abstract + claims into a dense vector once
          at index-build time.
        • At query time, encodes the query and retrieves the top-N candidates
          using L2 distance in FAISS (O(log n) with IVF indexes at scale).
        • Hybrid filters (classification prefix, title keyword, abstract
          keyword) are applied POST-retrieval on the candidate set rather
          than as a pre-filter. This is intentional: pre-filtering with FAISS
          IDSelector is correct but slower on small datasets; post-filtering
          over the top-N is effectively free and still accurate when N >> k.

    Phase 2 — Cross-encoder re-ranking (slow but precise)
        • Takes the filtered Phase-1 candidates and scores each
          (query, patent_text) pair with a cross-encoder.
        • Cross-encoders attend jointly to both texts, producing much more
          accurate relevance scores than bi-encoder cosine similarity.
        • Only run on the small candidate set, so latency stays acceptable.

    Why this two-phase approach?
        Bi-encoders are fast but treat query and document independently,
        which loses nuance. Cross-encoders are accurate but too slow to
        score every patent. The two-phase pattern (retrieve-then-rerank)
        is the standard production pattern used in systems like Bing, Google
        Patents, and most enterprise search stacks.
    """

    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)
        print('Loading bi-encoder ...')
        self.bi_encoder   = SentenceTransformer(BI_ENCODER_MODEL)
        print('Loading cross-encoder ...')
        self.cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
        self.index = None
        self.embeddings = None

    # ------------------------------------------------------------------
    # Index building
    # ------------------------------------------------------------------
    def build_index(self):
        """
        Encode all patents and add them to a FAISS index.

        Index choice: IndexFlatL2
            Exact nearest-neighbour search — appropriate for this dataset
            (~100–1000 patents). At 10^6+ patents, switch to IndexIVFFlat
            (inverted file index) for sub-linear query time:

                nlist = int(np.sqrt(len(df)))  # number of Voronoi cells
                quantiser = faiss.IndexFlatL2(dim)
                index = faiss.IndexIVFFlat(quantiser, dim, nlist)
                index.train(embeddings)        # must train before adding
                index.nprobe = 10              # cells to scan at query time
        """
        print('Encoding patent texts ...')
        t0 = time.time()
        self.embeddings = self.bi_encoder.encode(
            self.df['search_text'].tolist(),
            show_progress_bar=True,
            convert_to_numpy=True,
        ).astype('float32')

        dim = self.embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(self.embeddings)
        elapsed = time.time() - t0
        print(f'Index built: {self.index.ntotal:,} vectors, dim={dim}, time={elapsed:.1f}s')

    # ------------------------------------------------------------------
    # Core search
    # ------------------------------------------------------------------
    def search(
        self,
        query: str,
        top_k: int = DEFAULT_TOP_K,
        class_prefix: str = None,   # e.g. 'B60B' — matches classification prefix
        keyword: str = None,        # keyword searched in title + abstract
        rerank: bool = True,        # whether to apply cross-encoder re-ranking
        phase1_k: int = PHASE1_CANDIDATES,
    ) -> dict:
        """
        Execute a search and return a result dict with timing information.

        Parameters
        ----------
        query        : Natural-language search query.
        top_k        : Number of results to return.
        class_prefix : Optional CPC code prefix filter (e.g. 'B60B', 'B60C').
                       Matches patents whose classification starts with this string.
        keyword      : Optional keyword that must appear in title or abstract.
        rerank       : If True, apply cross-encoder re-ranking (Phase 2).
        phase1_k     : How many candidates to retrieve in Phase 1 before
                       filtering and re-ranking. Should be >> top_k.

        Returns
        -------
        dict with keys:
            results    : DataFrame of top_k patents
            timing     : dict of per-phase latencies in milliseconds
        """
        if self.index is None:
            raise RuntimeError('Call build_index() before searching.')

        timing = {}

        # ── Phase 1: bi-encoder + FAISS ──────────────────────────────────
        t0 = time.time()
        query_vec = self.bi_encoder.encode([query], convert_to_numpy=True).astype('float32')
        distances, indices = self.index.search(query_vec, min(phase1_k, len(self.df)))

        candidates = self.df.iloc[indices[0]].copy()
        candidates['_l2_dist'] = distances[0]
        timing['phase1_ms'] = (time.time() - t0) * 1000

        # ── Hybrid filters ────────────────────────────────────────────────
        t1 = time.time()
        if class_prefix:
            # Pre-parsed cpc_prefix column makes this O(n) string comparison
            # rather than a regex over the raw code — faster at scale.
            candidates = candidates[
                candidates['cpc_prefix'].str.startswith(class_prefix, na=False)
            ]
        if keyword:
            kw = keyword.lower()
            candidates = candidates[
                candidates['title'].str.lower().str.contains(kw, na=False) |
                candidates['abstract'].str.lower().str.contains(kw, na=False)
            ]
        timing['filter_ms'] = (time.time() - t1) * 1000

        if candidates.empty:
            return {'results': pd.DataFrame(), 'timing': timing}

        # ── Phase 2: cross-encoder re-ranking ─────────────────────────────
        if rerank and len(candidates) > 1:
            t2 = time.time()
            # Build (query, document) pairs — use title + abstract + first claim
            # (truncated) to keep inference fast while retaining key content
            pairs = [
                (query, f"{row['title']}. {row['abstract'][:400]}")
                for _, row in candidates.iterrows()
            ]
            ce_scores = self.cross_encoder.predict(pairs)
            candidates = candidates.copy()
            candidates['_ce_score'] = ce_scores
            candidates = candidates.sort_values('_ce_score', ascending=False)
            timing['rerank_ms'] = (time.time() - t2) * 1000
        else:
            # Fall back to bi-encoder L2 distance (lower = more similar)
            candidates = candidates.sort_values('_l2_dist', ascending=True)
            timing['rerank_ms'] = 0.0

        timing['total_ms'] = sum(timing.values())

        result_cols = ['doc_id', 'title', 'cpc_prefix', 'classification',
                       'abstract', '_l2_dist']
        if '_ce_score' in candidates.columns:
            result_cols.append('_ce_score')

        return {
            'results': candidates[result_cols].head(top_k).reset_index(drop=True),
            'timing' : timing,
        }


print('ThinkstructSearch class defined.')

ThinkstructSearch class defined.


## 4. Initialize engine & build index

In [17]:
engine = ThinkstructSearch(df_patents)
engine.build_index()

Loading bi-encoder ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Encoding patent texts ...


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Index built: 1,280 vectors, dim=384, time=175.4s


## 5. Helper — pretty-print results

In [18]:
def show_results(result: dict, query: str, label: str = ''):
    """Render search results in a readable format."""
    t = result['timing']
    tag = f'[{label}] ' if label else ''
    print(f'\n{tag}Query: "{query}"')
    print(
        f"  Phase 1 (FAISS):  {t.get('phase1_ms', 0):.1f} ms  |  "
        f"Filter:  {t.get('filter_ms', 0):.1f} ms  |  "
        f"Re-rank: {t.get('rerank_ms', 0):.1f} ms  |  "
        f"Total: {t.get('total_ms', 0):.1f} ms"
    )
    df = result['results']
    if df.empty:
        print('  No results found (try relaxing the filters).')
        return
    for i, row in df.iterrows():
        ce = f"  CE={row['_ce_score']:.3f}" if '_ce_score' in row else ''
        print(f"  {i+1}. [{row['doc_id']}] {row['title'][:70]}")
        print(f"      CPC: {row['classification']:15s}  L2={row['_l2_dist']:.3f}{ce}")
        print(f"      {row['abstract'][:120]}...")

## 6. Demo — query types

### 6a. Natural language query (standard semantic)

In [19]:
query = "innovative braking systems for electric vehicles"

res = engine.search(query, top_k=5, rerank=True)
show_results(res, query, label='Semantic + Re-rank')


[Semantic + Re-rank] Query: "innovative braking systems for electric vehicles"
  Phase 1 (FAISS):  65.0 ms  |  Filter:  0.0 ms  |  Re-rank: 15447.1 ms  |  Total: 15512.1 ms
  1. [20250091384] ELECTRIC VEHICLE MOTOR WITH DOUBLE DISC BRAKES
      CPC: B60B2108FI       L2=0.827  CE=2.322
      An electric vehicle motor with double disc brakes is provided, and relates to the field of electric vehicle motors. The ...
  2. [20250091384] ELECTRIC VEHICLE MOTOR WITH DOUBLE DISC BRAKES
      CPC: B60B2108FI       L2=0.827  CE=2.322
      An electric vehicle motor with double disc brakes is provided, and relates to the field of electric vehicle motors. The ...
  3. [20240270018] SYSTEMS AND METHODS FOR WHEEL DISCONNECT
      CPC: B60B2700FI       L2=1.311  CE=-0.888
      Systems are provided for a disconnect system for an electric vehicle. In one example, a system includes a disconnect dev...
  4. [20240270018] SYSTEMS AND METHODS FOR WHEEL DISCONNECT
      CPC: B60B2700FI       L2=1.311  CE=-

### 6b. Hybrid search — classification-code filter

In [20]:
# Restrict to B60B (wheels) patents only
res_hybrid = engine.search(query, top_k=5, class_prefix='B60B', rerank=True)
show_results(res_hybrid, query, label='Hybrid (B60B filter) + Re-rank')


[Hybrid (B60B filter) + Re-rank] Query: "innovative braking systems for electric vehicles"
  Phase 1 (FAISS):  35.3 ms  |  Filter:  2.5 ms  |  Re-rank: 5591.9 ms  |  Total: 5629.7 ms
  1. [20250091384] ELECTRIC VEHICLE MOTOR WITH DOUBLE DISC BRAKES
      CPC: B60B2108FI       L2=0.827  CE=2.322
      An electric vehicle motor with double disc brakes is provided, and relates to the field of electric vehicle motors. The ...
  2. [20250091384] ELECTRIC VEHICLE MOTOR WITH DOUBLE DISC BRAKES
      CPC: B60B2108FI       L2=0.827  CE=2.322
      An electric vehicle motor with double disc brakes is provided, and relates to the field of electric vehicle motors. The ...
  3. [20240270018] SYSTEMS AND METHODS FOR WHEEL DISCONNECT
      CPC: B60B2700FI       L2=1.311  CE=-0.888
      Systems are provided for a disconnect system for an electric vehicle. In one example, a system includes a disconnect dev...
  4. [20240270018] SYSTEMS AND METHODS FOR WHEEL DISCONNECT
      CPC: B60B2700FI       L2=1

### 6c. Hybrid search — keyword filter

In [21]:
res_kw = engine.search(query, top_k=5, keyword='tire', rerank=True)
show_results(res_kw, query, label='Hybrid (keyword=tire) + Re-rank')


[Hybrid (keyword=tire) + Re-rank] Query: "innovative braking systems for electric vehicles"
  Phase 1 (FAISS):  29.7 ms  |  Filter:  2.8 ms  |  Re-rank: 1725.8 ms  |  Total: 1758.2 ms
  1. [20240198723] Traction Enhancement and Improved Spokes for Airless Tires
      CPC: B60B928FI        L2=1.209  CE=-4.746
      The invention is embodied in a tire having retractable studs. A driver can activate the studs for driving in icy and/or ...
  2. [20240198723] Traction Enhancement and Improved Spokes for Airless Tires
      CPC: B60B928FI        L2=1.209  CE=-4.746
      The invention is embodied in a tire having retractable studs. A driver can activate the studs for driving in icy and/or ...
  3. [20240100881] AIRLESS TIRE
      CPC: B60C714FI        L2=1.356  CE=-8.789
      It is possible to maintain high-speed durability while having a function of eliminating static electricity of a vehicle....
  4. [20240100881] AIRLESS TIRE
      CPC: B60C714FI        L2=1.356  CE=-8.789
      It is p

### 6d. Claim-text as input (paste a claim directly)

In [22]:
claim_query = (
    "A wheel assembly comprising a rim and a plurality of spokes, "
    "wherein each spoke includes an injection-molded connecting element "
    "with a friction-enhancing surface structure."
)

res_claim = engine.search(claim_query, top_k=5, rerank=True)
show_results(res_claim, claim_query[:60] + '...', label='Claim-text input')


[Claim-text input] Query: "A wheel assembly comprising a rim and a plurality of spokes,..."
  Phase 1 (FAISS):  45.7 ms  |  Filter:  0.0 ms  |  Re-rank: 7055.6 ms  |  Total: 7101.2 ms
  1. [20240343062] WHEEL ASSEMBLY FOR THREE-WHEELED VEHICLE
      CPC: B60B926FI        L2=0.645  CE=4.286
      A wheel assembly includes a rim and a hub concentric with the rim. The wheel assembly includes a first set of spokes spa...
  2. [20240343062] WHEEL ASSEMBLY FOR THREE-WHEELED VEHICLE
      CPC: B60B926FI        L2=0.645  CE=4.286
      A wheel assembly includes a rim and a hub concentric with the rim. The wheel assembly includes a first set of spokes spa...
  3. [20240208262] WHEEL COMPONENT, METHOD OF MANUFACTURING, AND TOOL DEVICE
      CPC: B60B100FI        L2=0.693  CE=3.966
      A wheel component, method, and tool device for a bicycle, including a plurality of integrally interconnected component p...
  4. [20240208262] WHEEL COMPONENT, METHOD OF MANUFACTURING, AND TOOL DEVICE
      CPC:

### 6e. Search by patent ID — find similar patents to a known one

In [23]:
def search_by_patent_id(patent_id: str, top_k: int = 5, **kwargs) -> dict:
    """
    Find patents similar to a given patent (by doc_id).
    Uses that patent's own abstract + claims as the query.
    The source patent itself is excluded from results.
    """
    row = engine.df[engine.df['doc_id'] == patent_id]
    if row.empty:
        raise ValueError(f'Patent {patent_id} not found in the loaded dataset.')
    query_text = row.iloc[0]['search_text']
    result = engine.search(query_text, top_k=top_k + 1, **kwargs)
    # Remove the source patent from results
    result['results'] = (
        result['results'][result['results']['doc_id'] != patent_id]
        .head(top_k)
        .reset_index(drop=True)
    )
    return result


# Use the first patent in the dataset as the reference
ref_id = df_patents.iloc[0]['doc_id']
print(f'Finding patents similar to: {ref_id} — "{df_patents.iloc[0]["title"]}"')
res_by_id = search_by_patent_id(ref_id, top_k=5, rerank=True)
show_results(res_by_id, f'Similar to {ref_id}', label='Patent-ID input')

Finding patents similar to: 20240051333 — "SPOKE"

[Patent-ID input] Query: "Similar to 20240051333"
  Phase 1 (FAISS):  135.9 ms  |  Filter:  0.0 ms  |  Re-rank: 30860.5 ms  |  Total: 30996.4 ms
  1. [20250033414] SPOKE FOR NON-PNEUMATIC TIRE
      CPC: B60C714FI        L2=0.815  CE=-0.699
      A non-pneumatic wheel ( 10 ) having a hub ( 12 ), an outer tread band ( 200 ) and a plurality of spokes ( 100 ) connecti...
  2. [20250033414] SPOKE FOR NON-PNEUMATIC TIRE
      CPC: B60C714FI        L2=0.815  CE=-0.699
      A non-pneumatic wheel ( 10 ) having a hub ( 12 ), an outer tread band ( 200 ) and a plurality of spokes ( 100 ) connecti...
  3. [20240308267] Cycle Hub Assembly
      CPC: B60B104FI        L2=0.796  CE=-0.757
      The invention relates to a cycle hub assembly and a method of producing such cycle hub assembly. The cycle hub assembly ...
  4. [20240308267] Cycle Hub Assembly
      CPC: B60B104FI        L2=0.796  CE=-0.757
      The invention relates to a cycle hub assembl

## 7. Enhancement 1 — Hybrid search performance benchmark

> **Requirement:** Time the algorithm with and without hybrid filtering and comment on efficiency.


In [25]:
BENCHMARK_QUERIES = [
    "wheel rim with lightweight composite material",
    "tire pressure monitoring sensor",
    "vehicle suspension shock absorber",
    "spoke with injection molded connector",
    "tread pattern for all-season tires",
]

N_RUNS = 5  # average over multiple runs for stable timings

rows = []
for q in BENCHMARK_QUERIES:
    # ── Semantic only (no filter, no rerank) ────────────────────────────
    times_sem = []
    for _ in range(N_RUNS):
        r = engine.search(q, top_k=5, rerank=False)
        times_sem.append(r['timing'].get('total_ms', 0.0))

    # ── Hybrid with classification filter (no rerank) ───────────────────
    times_hyb = []
    for _ in range(N_RUNS):
        r = engine.search(q, top_k=5, class_prefix='B60B', rerank=False)
        times_hyb.append(r['timing'].get('total_ms', 0.0))

    # ── Full pipeline: hybrid + cross-encoder re-rank ───────────────────
    times_full = []
    for _ in range(N_RUNS):
        r = engine.search(q, top_k=5, class_prefix='B60B', rerank=True)
        times_full.append(r['timing'].get('total_ms', 0.0))

    rows.append({
        'query'               : q[:45],
        'semantic_ms'         : round(np.mean(times_sem), 1),
        'hybrid_ms'           : round(np.mean(times_hyb), 1),
        'hybrid_rerank_ms'    : round(np.mean(times_full), 1),
    })

bench_df = pd.DataFrame(rows)
print('\n=== Performance Benchmark (avg over 5 runs) ===\n')
display(bench_df.to_string(index=False))

print('\n--- Commentary ---')
print("""
Observations
------------
1. At this dataset scale (~100 patents) hybrid filtering adds negligible overhead
   because the post-FAISS filter is just a pandas boolean mask — effectively O(N)
   where N = phase1_k (100 candidates), not the full index.

2. The cross-encoder re-rank is the dominant cost. It runs a transformer forward
   pass for each (query, doc) pair — scaling linearly with the candidate count.
   On CPU with 100 candidates this is ~0.5–2 s depending on hardware.

Scalability improvements for 10^7 patents
-----------------------------------------
• FAISS IVFFlat / HNSW index: sub-linear ANN search, ~10–50x faster than Flat.
• Pre-filter before FAISS using an inverted index on CPC codes (e.g. Elasticsearch
  or a dict mapping prefix -> row indices), then pass only those indices to FAISS
  via IDSelectorBatch — avoids scanning unrelated partitions.
• Reduce rerank candidates: phase1_k=50 instead of 100 cuts cross-encoder cost ~2x
  with small precision loss.
• Batch GPU inference for the cross-encoder (device='cuda') gives ~5–10x speedup.
• Cache embeddings to disk (numpy .npy) so index rebuilds are instant on restart.
""")


=== Performance Benchmark (avg over 5 runs) ===



'                                        query  semantic_ms  hybrid_ms  hybrid_rerank_ms\nwheel rim with lightweight composite material         19.7       19.0            4221.3\n              tire pressure monitoring sensor         16.5       16.7             401.2\n            vehicle suspension shock absorber         26.8       26.9            2446.6\n        spoke with injection molded connector         23.8       25.0            4356.1\n           tread pattern for all-season tires         17.3        0.0               0.0'


--- Commentary ---

Observations
------------
1. At this dataset scale (~100 patents) hybrid filtering adds negligible overhead
   because the post-FAISS filter is just a pandas boolean mask — effectively O(N)
   where N = phase1_k (100 candidates), not the full index.

2. The cross-encoder re-rank is the dominant cost. It runs a transformer forward
   pass for each (query, doc) pair — scaling linearly with the candidate count.
   On CPU with 100 candidates this is ~0.5–2 s depending on hardware.

Scalability improvements for 10^7 patents
-----------------------------------------
• FAISS IVFFlat / HNSW index: sub-linear ANN search, ~10–50x faster than Flat.
• Pre-filter before FAISS using an inverted index on CPC codes (e.g. Elasticsearch
  or a dict mapping prefix -> row indices), then pass only those indices to FAISS
  via IDSelectorBatch — avoids scanning unrelated partitions.
• Reduce rerank candidates: phase1_k=50 instead of 100 cuts cross-encoder cost ~2x
  with small precision 

## 8. Enhancement 2 — Two-phase re-ranking quality comparison

In [26]:
def compare_rerank(query: str, top_k: int = 5):
    """
    Side-by-side: bi-encoder only vs bi-encoder + cross-encoder re-rank.
    Shows how result ordering changes after re-ranking.
    """
    res_no  = engine.search(query, top_k=top_k, rerank=False)
    res_yes = engine.search(query, top_k=top_k, rerank=True)

    print(f'Query: "{query}"\n')
    print(f'{"Rank":<5} {"── Bi-encoder only ──":<45} {"── + Cross-encoder re-rank ──":<45}')
    print('-' * 100)

    for i in range(top_k):
        def fmt(row):
            if row is None:
                return '(no result)'
            return f"{row['doc_id']}  {row['title'][:30]}"

        r_no  = res_no['results'].iloc[i]  if i < len(res_no['results'])  else None
        r_yes = res_yes['results'].iloc[i] if i < len(res_yes['results']) else None
        print(f'{i+1:<5} {fmt(r_no):<45} {fmt(r_yes):<45}')

    print(f"\nLatency — bi-encoder only: {res_no['timing']['total_ms']:.1f} ms  "
          f"| with re-rank: {res_yes['timing']['total_ms']:.1f} ms")


compare_rerank("lightweight composite wheel rim with spoke attachment")

Query: "lightweight composite wheel rim with spoke attachment"

Rank  ── Bi-encoder only ──                         ── + Cross-encoder re-rank ──                
----------------------------------------------------------------------------------------------------
1     20240270012  ULTRA-LIGHTWEIGHT STEEL WHEEL    20240278595  FACE TO RIM REINFORCING CONNEC  
2     20240270012  ULTRA-LIGHTWEIGHT STEEL WHEEL    20240278595  FACE TO RIM REINFORCING CONNEC  
3     20250115075  RIM SPOKE                        20250018742  Wheel with Flexible Wide-Body   
4     20250115075  RIM SPOKE                        20250018742  Wheel with Flexible Wide-Body   
5     20240217263  WHEEL                            20240316985  Wheel With High Strength Flexi  

Latency — bi-encoder only: 26.0 ms  | with re-rank: 5033.7 ms


## 9. Interactive search cell

Edit the variables below and re-run this cell to search interactively.

In [27]:
# ── Edit these to run a custom search ─────────────────────────────────────
MY_QUERY        = "tire with improved grip and puncture resistance"
MY_CLASS_PREFIX = None      # e.g. 'B60B', 'B60C', or None for no filter
MY_KEYWORD      = None      # e.g. 'rubber', 'carbon fiber', or None
MY_TOP_K        = 5
USE_RERANK      = True
# ──────────────────────────────────────────────────────────────────────────

r = engine.search(
    MY_QUERY,
    top_k=MY_TOP_K,
    class_prefix=MY_CLASS_PREFIX,
    keyword=MY_KEYWORD,
    rerank=USE_RERANK,
)
show_results(r, MY_QUERY, label='Custom search')


[Custom search] Query: "tire with improved grip and puncture resistance"
  Phase 1 (FAISS):  24.0 ms  |  Filter:  0.0 ms  |  Re-rank: 6743.7 ms  |  Total: 6767.7 ms
  1. [20240208269] PNEUMATIC TYRE FOR A TWO-WHEELED VEHICLE HAVING A PROTECTIVE LAYER
      CPC: B60C909FI        L2=1.128  CE=4.199
      A two-wheeler pneumatic tire, in which damage protection and puncture protection are improved while providing good rolli...
  2. [20240208269] PNEUMATIC TYRE FOR A TWO-WHEELED VEHICLE HAVING A PROTECTIVE LAYER
      CPC: B60C909FI        L2=1.128  CE=4.199
      A two-wheeler pneumatic tire, in which damage protection and puncture protection are improved while providing good rolli...
  3. [20250010665] TIRE WITH HIGH COMPRESSION SET ELASTOMER HAVING IMPROVED WEAR PERFORMA
      CPC: B60C1112FI       L2=1.036  CE=3.597
      A tread design for a tire is described herein that has excellent wear properties, hydroplaning resistance, and snow grip...
  4. [20250010665] TIRE WITH HIGH COMPRES